## ☕️ Coffee Production Prediction

Given *data about coffee*, let's try to predict the **average production** of coffee in a given country.

We will use a random forest regression model to make our predictions.

Data source: https://www.kaggle.com/datasets/yamaerenay/ico-coffee-dataset-worldwide

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor

In [2]:
df_paths = [
    'archive/domestic-consumption.csv',
    'archive/exports-calendar-year.csv',
    'archive/exports-crop-year.csv',
    'archive/gross-opening-stocks.csv',
    'archive/total-production.csv'
]

In [4]:
dfs = [pd.read_csv(path) for path in df_paths]

In [12]:
dfs[0].mean(axis=1, numeric_only=True)

0        25.689655
1        41.103448
2     15234.310345
3         1.891966
4       214.137931
5      2662.137931
6       325.405724
7         1.310345
8         2.004828
9        19.482759
10      222.413793
11        1.195379
12        0.152069
13        4.802276
14        2.977034
15      213.244828
16      356.438724
17      333.275862
18     1501.310345
19       40.646586
20        0.570069
21        0.000000
22       79.787897
23       13.560000
24     1388.097669
25      370.238621
26      224.850655
27      200.000000
28      229.725310
29        0.000000
30     2529.034483
31        0.628241
32        5.661379
33      326.034483
34       48.120690
35        4.978759
36      259.109586
37     1153.137931
38       10.429897
39       52.241379
40       85.379310
41        4.793103
42     1749.517241
43        0.000000
44      165.000172
45       39.275862
46       68.331586
47        5.620690
48       32.298862
49      669.310345
50        1.236793
51       11.775862
52      154.

In [15]:
def get_means(df):
    df = df.copy()
    countries = df[df.columns[0]]
    means = df.mean(axis=1, numeric_only=True)
    df = pd.concat([countries, means], axis=1)
    df.columns = ['country', countries.name]
    return df

In [16]:
get_means(dfs[0])

,country,domestic_consumption
0,Angola,25.689655
1,Bolivia (Plurinational State of),41.103448
2,Brazil,15234.310345
3,Burundi,1.891966
4,Ecuador,214.137931
5,Indonesia,2662.137931
6,Madagascar,325.405724
7,Malawi,1.310345
8,Papua New Guinea,2.004828
9,Paraguay,19.482759


In [18]:
def make_df(dfs):
    # Process all dataframes
    processed_dfs = []

    for df in dfs:
        processed_dfs.append(get_means(df))

    # Merge DataFrames 
    df = processed_dfs[0]

    for i in range(1, len(processed_dfs)):
        df = df.merge(processed_dfs[i], on='country')

    return df

In [19]:
data = make_df(dfs)

In [20]:
data

,country,domestic_consumption,exports,exports_crop_year,gross_opening_stocks,total_production
0,Angola,25.689655,24.115531,23.187083,42.068966,45.256048
1,Bolivia (Plurinational State of),41.103448,78.384152,76.904748,7.237931,116.146128
2,Brazil,15234.310345,25706.195606,25919.128803,23213.206897,41067.783976
3,Burundi,1.891966,363.186423,356.056321,48.979690,350.172424
4,Ecuador,214.137931,1115.800914,1105.434110,108.592069,1076.504352
5,Indonesia,2662.137931,5878.047357,5879.061059,690.114655,8452.302438
6,Madagascar,325.405724,289.048949,280.822603,87.587000,587.719424
7,Malawi,1.310345,47.247465,46.095669,4.896552,46.716359
8,Papua New Guinea,2.004828,1015.455512,1015.019583,55.757928,1010.334755
9,Paraguay,19.482759,25.757542,20.850776,36.379310,34.678362


### Preprocessing

In [22]:
# Drop country column
df = data.drop('country', axis=1)

In [24]:
# Split df into X and y
y = df['total_production'].copy()
X = df.drop('total_production', axis=1).copy()

In [25]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

In [26]:
X_train

,domestic_consumption,exports,exports_crop_year,gross_opening_stocks
52,154.928966,3080.789345,3132.876862,1049.680310
35,4.978759,1.305947,1.318552,0.000000
26,224.850655,2640.989406,2593.019476,1027.788241
45,39.275862,6.434295,6.366579,0.275862
54,1016.050759,13048.102878,13701.263769,1483.241414
27,200.000000,438.641994,393.192238,34.143379
34,48.120690,225.830006,226.816866,26.413828
13,4.802276,78.112883,76.360955,18.070241
22,79.787897,795.942577,729.091645,181.943069
47,5.620690,50.098006,46.814148,14.310345


In [27]:
X_test

,domestic_consumption,exports,exports_crop_year,gross_opening_stocks
44,165.000172,1270.768988,1329.936879,86.193345
2,15234.310345,25706.195606,25919.128803,23213.206897
46,68.331586,91.047137,89.761421,31.493793
19,40.646586,786.307663,789.624797,75.256400
32,5.661379,30.461676,30.591972,12.370448
33,326.034483,3590.512097,3614.588628,106.695931
36,259.109586,3194.853765,3355.469269,153.354034
39,52.241379,989.200405,959.627121,203.535276
49,669.310345,551.847711,521.790931,84.168414
42,1749.517241,3143.855086,3109.249886,598.732759


In [28]:
# Scale X
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), index=X_train.index, columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

### Training / Results

In [29]:
model = RandomForestRegressor()
model.fit(X_train, y_train)
print("Model Trained.")

Model Trained.


In [31]:
y_pred = model.predict(X_test)

rmse = np.sqrt(np.mean((y_test - y_pred)**2))

print("RMSE: {:.2f}".format(rmse))

RMSE: 7509.49


In [34]:
r2_score = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)

print("R2 Score: {:.4f}".format(r2_score))

R2 Score: 0.3929
